# BMIN 5200 — Week 3 in-class exercise
## Traversing a biomedical ontology

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week03.ipynb)

**Time:** ~25 minutes · **Pairs with:** Semantic networks, frames, and ontologies (is-a hierarchies, inheritance, inferential distance, GO / RxNorm / MeSH / SNOMED CT)

### Tasks
- Build a small anatomy-and-disease ontology in `networkx` with two relations, `is_a` and `part_of`
- Implement inheritance of default properties down the is-a chain, and ancestor / descendant queries
- Resolve conflicting inherited defaults with the inferential-distance rule, and find the case where it cannot decide
- See why `part_of` does **not** inherit the way `is_a` does, then look at the same structure in RxNorm through the live RxNav API

### Background
Every terminology a health system runs on — SNOMED CT for problems, RxNorm for medications,
LOINC for labs, GO for gene function — is a graph of exactly this shape, and every query that
"rolls up" a patient cohort or a drug class is doing the traversal you are about to write.
The two mistakes in this notebook, treating `part_of` like `is_a` and silently picking one of
two conflicting inherited defaults, are both live sources of wrong cohort counts and wrong
decision-support alerts in production systems today.

## Setup

Everything here is preinstalled in Colab. The `requests` import is only used in Part 4, and
that section falls back to a bundled copy of the response if the network is unavailable, so
this notebook runs offline.

In [ ]:
from collections import deque

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import requests

print("Setup complete. networkx", nx.__version__)

## Part 1 — The ontology

Below is a hand-written slice of an anatomy-plus-disease ontology: 35 concepts, two
relations. It is modeled on how SNOMED CT and the Gene Ontology are actually built, but it is
tiny and simplified on purpose — a real `.obo` file for GO alone has about 40,000 terms, and
downloading one would eat the whole class.

Edges point from the **more specific concept to the more general one**, so the tuple
`("small_cell_lung_carcinoma", "lung_carcinoma")` reads left to right as a sentence. Node
attributes are *frame slots*: default property values attached to a concept, which more
specific concepts inherit.

In [ ]:
IS_A = [
    # anatomy
    ("organ_system", "anatomical_structure"),
    ("organ", "anatomical_structure"),
    ("tissue", "anatomical_structure"),
    ("cell", "anatomical_structure"),
    ("airway", "anatomical_structure"),
    ("lung_lobe", "anatomical_structure"),
    ("alveolus", "anatomical_structure"),
    ("respiratory_system", "organ_system"),
    ("lung", "organ"),
    ("left_lung", "lung"),
    ("right_lung", "lung"),
    ("upper_lobe_of_left_lung", "lung_lobe"),
    ("lower_lobe_of_left_lung", "lung_lobe"),
    ("bronchus", "airway"),
    ("trachea", "airway"),
    ("bronchial_epithelium", "tissue"),
    ("pulmonary_neuroendocrine_cell", "cell"),
    # disease
    ("neoplasm", "disease"),
    ("respiratory_disease", "disease"),
    ("lung_disease", "respiratory_disease"),
    ("lung_neoplasm", "lung_disease"),
    ("lung_neoplasm", "neoplasm"),
    ("lung_carcinoma", "lung_neoplasm"),
    ("non_small_cell_lung_carcinoma", "lung_carcinoma"),
    ("lung_adenocarcinoma", "non_small_cell_lung_carcinoma"),
    ("lung_squamous_cell_carcinoma", "non_small_cell_lung_carcinoma"),
    ("neuroendocrine_tumor", "neoplasm"),
    ("small_cell_lung_carcinoma", "lung_carcinoma"),
    ("small_cell_lung_carcinoma", "neuroendocrine_tumor"),
    ("combined_small_cell_lung_carcinoma", "small_cell_lung_carcinoma"),
    ("obstructive_lung_disease", "lung_disease"),
    ("copd", "obstructive_lung_disease"),
    ("asthma", "obstructive_lung_disease"),
    ("pneumonia", "lung_disease"),
]

PART_OF = [
    ("lower_respiratory_tract", "respiratory_system"),
    ("lung", "lower_respiratory_tract"),
    ("trachea", "lower_respiratory_tract"),
    ("bronchus", "lower_respiratory_tract"),
    ("upper_lobe_of_left_lung", "left_lung"),
    ("lower_lobe_of_left_lung", "left_lung"),
    ("alveolus", "lung"),
    ("bronchial_epithelium", "bronchus"),
    ("pulmonary_neuroendocrine_cell", "bronchial_epithelium"),
]

# Frame slots: a default that every subclass inherits unless it says otherwise.
SLOTS = {
    "anatomical_structure": {"has_spatial_extent": True},
    "lung": {"typical_volume_ml": 6000, "resectable_unit": True},
    "bronchus": {"resectable_unit": False},
    "disease": {"managed_by": "primary_care"},
    "respiratory_disease": {"managed_by": "pulmonology"},
    "lung_disease": {"site": "lung", "first_imaging": "chest_radiograph"},
    "lung_neoplasm": {"first_imaging": "chest_ct"},
    "lung_carcinoma": {"first_line_treatment": "surgical_resection",
                       "staging_system": "TNM_8"},
    "neuroendocrine_tumor": {"staging_system": "ENETS_grading",
                             "marker_panel": "chromogranin_and_synaptophysin"},
    "small_cell_lung_carcinoma": {"first_line_treatment": "platinum_etoposide_plus_radiation"},
    "obstructive_lung_disease": {"first_test": "spirometry"},
}

ontology = nx.DiGraph()
for child, parent in IS_A:
    ontology.add_edge(child, parent, relation="is_a")
for part, whole in PART_OF:
    ontology.add_edge(part, whole, relation="part_of")
for concept, slots in SLOTS.items():
    ontology.nodes[concept].update(slots)

print(f"{ontology.number_of_nodes()} concepts, "
      f"{len(IS_A)} is_a edges, {len(PART_OF)} part_of edges")

Two helpers do all the traversal. `linked_up` returns the concepts one edge above a node
along a chosen relation; `ancestors_of` keeps following those edges to the top. Both are
written for you. Note that `small_cell_lung_carcinoma` has **two** is-a parents — it is a
lung carcinoma and it is a neuroendocrine tumor — which is normal in SNOMED CT and is where
the trouble in Part 3 comes from.

In [ ]:
def linked_up(concept, relation):
    """Concepts exactly one `relation` edge above this one."""
    return [target for _, target, data in ontology.out_edges(concept, data=True)
            if data["relation"] == relation]


def linked_down(concept, relation):
    """Concepts exactly one `relation` edge below this one."""
    return [source for source, _, data in ontology.in_edges(concept, data=True)
            if data["relation"] == relation]


def ancestors_of(concept, relation="is_a"):
    """Everything above `concept`, breadth-first, so nearer ancestors come first."""
    found, queue = [], deque(linked_up(concept, relation))
    while queue:
        current = queue.popleft()
        if current in found:
            continue
        found.append(current)
        queue.extend(linked_up(current, relation))
    return found


print("is_a parents of small_cell_lung_carcinoma:", linked_up("small_cell_lung_carcinoma", "is_a"))
print()
print("everything small_cell_lung_carcinoma is a kind of:")
for ancestor in ancestors_of("small_cell_lung_carcinoma"):
    print("   ", ancestor)

Here is the shape of that neighborhood drawn out. Small cell carcinoma sits underneath two
independent branches of the hierarchy, and both branches carry opinions about how to treat
and stage it.

In [ ]:
diamond = ["neoplasm", "lung_neoplasm", "neuroendocrine_tumor", "lung_carcinoma",
           "non_small_cell_lung_carcinoma", "small_cell_lung_carcinoma",
           "combined_small_cell_lung_carcinoma"]
positions = {
    "neoplasm": (1.6, 4), "lung_neoplasm": (0.6, 3), "neuroendocrine_tumor": (3.2, 3),
    "lung_carcinoma": (0.9, 2), "non_small_cell_lung_carcinoma": (-0.4, 1),
    "small_cell_lung_carcinoma": (2.3, 1), "combined_small_cell_lung_carcinoma": (2.3, 0),
}
short_names = {
    "neoplasm": "neoplasm", "lung_neoplasm": "lung\nneoplasm",
    "neuroendocrine_tumor": "neuroendocrine\ntumor", "lung_carcinoma": "lung\ncarcinoma",
    "non_small_cell_lung_carcinoma": "NSCLC", "small_cell_lung_carcinoma": "SCLC",
    "combined_small_cell_lung_carcinoma": "combined\nSCLC",
}

plt.figure(figsize=(7, 5))
nx.draw_networkx(ontology.subgraph(diamond), pos=positions, labels=short_names,
                 node_size=3000, font_size=8, arrows=True)
plt.title("is_a edges point from the specific concept to the general one")
plt.axis("off")
plt.show()

## Part 2 — Inheritance and non-transitive relations

Two exercises here. The first is a warm-up: `descendants_of` currently returns only the
concepts one hop below, so asking for everything under `lung_carcinoma` misses the leaves.
Make it go all the way down — it is the mirror image of `ancestors_of` above.

In [ ]:
def descendants_of(concept, relation="is_a"):
    """Everything below `concept` -- every kind of thing it covers."""
    # TODO: right now this only looks one hop down. Rewrite it the way `ancestors_of`
    #       works, but using linked_down, so it reaches the leaves.
    return linked_down(concept, relation)


print("kinds of lung_carcinoma:", sorted(descendants_of("lung_carcinoma")))
print()
print("every concept that would be counted in a 'lung disease' cohort:")
print(sorted(descendants_of("lung_disease")))

That second query is the whole point of an is-a hierarchy in practice. A cohort defined as
"patients with lung disease" means patients coded with **any descendant** of `lung_disease`,
and a query that stops one hop short silently drops every patient coded with a specific
diagnosis. This is the *Is-a Hierarchy* and *Inference through Inheritance* material from
lecture, and it is also the single most common reason two analysts get different denominators
from the same EHR.

Now the main exercise. `inherit` looks for a slot on the concept itself, then on its
immediate parents, and gives up. Make it keep walking up until it finds the slot or runs out
of graph. Note the `relations` argument: the caller chooses which kinds of edge the search is
allowed to follow, and we are about to use that to make a point.

In [ ]:
def inherit(concept, slot, relations=("is_a",)):
    """Find `slot` for `concept`, returning (value, the concept it came from)."""
    if slot in ontology.nodes[concept]:
        return ontology.nodes[concept][slot], concept

    for relation in relations:
        for parent in linked_up(concept, relation):
            if slot in ontology.nodes[parent]:
                return ontology.nodes[parent][slot], parent

    # TODO: this stops after the immediate parents, so anything further up is invisible.
    #       Make the search continue upward -- the shortest fix is to call inherit()
    #       on each parent and return the first non-empty answer.
    return None, None


for concept, slot in [("lung_adenocarcinoma", "first_imaging"),
                      ("pneumonia", "managed_by"),
                      ("asthma", "managed_by")]:
    value, source = inherit(concept, slot)
    print(f"{concept:32s} {slot:16s} -> {str(value):22s} (from {source})")

### Predict before you run

`asthma` has no `managed_by` slot of its own. Going up the is-a chain, the first ancestor
that does is `respiratory_disease`, and above that `disease` says `primary_care`. Before you
fix the TODO, commit to two numbers out loud: **how many hops** up from `asthma` to the
answer, and **which** of the two values you should get.

Then fix `inherit` and re-run the cell above. The answer that comes back is the one an
inheritance-based decision support tool would put in front of a clinician.

### Why `part_of` is not `is_a`

Both relations look like arrows going up, and it is tempting to write one traversal that
follows whichever edge it finds. Run this. `alveolus` has no volume of its own; the search
that is allowed to cross `part_of` edges finds one anyway.

In [ ]:
comparison = []
for concept, slot in [("alveolus", "typical_volume_ml"),
                      ("alveolus", "resectable_unit"),
                      ("upper_lobe_of_left_lung", "typical_volume_ml"),
                      ("lung_adenocarcinoma", "first_imaging")]:
    strict_value, strict_source = inherit(concept, slot, relations=("is_a",))
    loose_value, loose_source = inherit(concept, slot, relations=("is_a", "part_of"))
    comparison.append({
        "concept": concept,
        "slot": slot,
        "is_a only": f"{strict_value} (from {strict_source})",
        "is_a + part_of": f"{loose_value} (from {loose_source})",
    })

print(pd.DataFrame(comparison).to_string(index=False))

An alveolus is part of a lung, and a lung holds about 6,000 mL. An alveolus holds about
0.004 mL. The loose traversal is off by six orders of magnitude, and it will also tell you
that an alveolus is a thing a surgeon can resect. Once you have made `inherit` recursive it
gets worse: the upper lobe of the left lung comes back with the volume of a whole lung.
`is_a` licenses inheritance because every instance of the child *is* an instance of the
parent; `part_of` licenses nothing of the kind, because a part is not an instance of its
whole.

What `part_of` does support is **location**: if a finding is in the alveolus, it is in the
lung, and in the respiratory system. The Gene Ontology relies on exactly this — its "true
path rule" propagates annotations up both `is_a` and `part_of`, because *being annotated to a
part* really does imply *involvement in the whole*. The relation is transitive; it just is not
inheritance.

In [ ]:
def containing_structures(concept):
    """Follow part_of upward: where in the body is this thing?"""
    return ancestors_of(concept, relation="part_of")


print("a pulmonary neuroendocrine cell is located in:")
for structure in containing_structures("pulmonary_neuroendocrine_cell"):
    print("   ", structure)
print()
print("Small cell carcinoma arises from these cells, which is why it presents centrally,")
print("near the hilum -- a fact this graph gets right through part_of and could not get")
print("through is_a.")

## Part 3 — Multiple inheritance and inferential distance

`small_cell_lung_carcinoma` inherits from `lung_carcinoma` and from `neuroendocrine_tumor`.
The inferential-distance rule says: when several ancestors supply a value for the same slot,
prefer the one from the **more specific** ancestor — the one closer to the concept you asked
about. `candidate_values` below collects every answer with its distance. Your job is to pick
the winner, and to notice when there is no winner.

In [ ]:
def candidate_values(concept, slot):
    """Every value for `slot` reachable up the is-a chain, with its distance."""
    found, queue, seen = [], deque([(concept, 0)]), set()
    while queue:
        current, distance = queue.popleft()
        if current in seen:
            continue
        seen.add(current)
        if slot in ontology.nodes[current]:
            found.append((distance, current, ontology.nodes[current][slot]))
        for parent in linked_up(current, "is_a"):
            queue.append((parent, distance + 1))
    return sorted(found)


def resolve(concept, slot):
    found = candidate_values(concept, slot)
    if not found:
        return {"value": None, "source": None, "distance": None, "ambiguous": False}

    distance, source, value = found[0]
    # TODO: `found` is sorted by distance, so found[0] is a nearest candidate -- but there
    #       may be ANOTHER candidate at the same distance carrying a DIFFERENT value.
    #       Set ambiguous=True when that happens instead of silently taking the first.
    return {"value": value, "source": source, "distance": distance, "ambiguous": False}


for concept, slot in [("lung_squamous_cell_carcinoma", "first_line_treatment"),
                      ("small_cell_lung_carcinoma", "first_line_treatment"),
                      ("combined_small_cell_lung_carcinoma", "first_line_treatment"),
                      ("small_cell_lung_carcinoma", "staging_system"),
                      ("combined_small_cell_lung_carcinoma", "staging_system")]:
    answer = resolve(concept, slot)
    flag = "  <-- AMBIGUOUS" if answer["ambiguous"] else ""
    print(f"{concept:36s} {slot:20s} -> {str(answer['value']):36s} "
          f"(distance {answer['distance']}, from {answer['source']}){flag}")
    for distance, source, value in candidate_values(concept, slot):
        print(f"{'':36s} {'':20s}    candidate: {value} at distance {distance} from {source}")
    print()

The `first_line_treatment` rows are inferential distance working as advertised. Squamous cell
carcinoma inherits `surgical_resection` from `lung_carcinoma`. Small cell carcinoma overrides
it locally with chemoradiation, and `combined_small_cell_lung_carcinoma` — one hop further
down — gets the override rather than the more distant default. That is clinically correct:
small cell disease is usually not resected, and a system that inherited "surgical resection"
from the general carcinoma node would be recommending an operation most of these patients
should not have.

`staging_system` is the case the rule cannot fix. `lung_carcinoma` says TNM and
`neuroendocrine_tumor` says ENETS grading; both sit at distance 1, neither is a subclass of
the other, and there is no principled way for the traversal to choose. Until you fill in the
TODO the function picks one silently — which is exactly the failure mode to be afraid of,
because the output looks like an answer.

One honest caveat, since this is a graduate course: what we implemented is the *path-length*
approximation to inferential distance. Touretzky's actual formulation compares specificity —
value A beats value B when A's source is a subclass of B's source — which is not always the
same as being fewer hops away in a tangled hierarchy. On this small graph the two agree. On
SNOMED CT, which has hundreds of thousands of concepts and heavy multiple inheritance, they
do not always, and description-logic reasoners are used instead of ad-hoc traversal.

## Part 4 — RxNorm via the RxNav API

RxNorm is the NLM's normalized drug terminology, and RxNav is its public REST API — no key,
no login. Its concepts are arranged in exactly the layers you just walked: an ingredient
(`IN`) such as warfarin, a precise ingredient (`PIN`) such as warfarin sodium, a clinical
drug (`SCD`) such as warfarin sodium 5 MG oral tablet, and a branded drug (`SBD`) such as the
same tablet marketed as Coumadin.

The cell below calls the live API. If the network is unavailable it prints a message and uses
a bundled copy of the response captured earlier, so the notebook runs either way.

In [ ]:
RXNAV_BASE = "https://rxnav.nlm.nih.gov/REST"

# Captured from the live API so this notebook works with no network. Trimmed to a few
# concepts per term type; the live call returns considerably more.
BUNDLED_RESPONSES = {
    f"{RXNAV_BASE}/rxcui.json?name=warfarin": {"idGroup": {"rxnormId": ["11289"]}},
    f"{RXNAV_BASE}/rxcui/11289/related.json?tty=IN+PIN+SCD+SBD": {"relatedGroup": {
        "rxcui": "11289",
        "conceptGroup": [
            {"tty": "IN", "conceptProperties": [
                {"rxcui": "11289", "name": "warfarin", "tty": "IN"}]},
            {"tty": "PIN", "conceptProperties": [
                {"rxcui": "114194", "name": "warfarin sodium", "tty": "PIN"},
                {"rxcui": "82118", "name": "warfarin potassium", "tty": "PIN"}]},
            {"tty": "SCD", "conceptProperties": [
                {"rxcui": "855288", "name": "warfarin sodium 1 MG Oral Tablet", "tty": "SCD"},
                {"rxcui": "855296", "name": "warfarin sodium 10 MG Oral Tablet", "tty": "SCD"},
                {"rxcui": "855302", "name": "warfarin sodium 2 MG Oral Tablet", "tty": "SCD"},
                {"rxcui": "855312", "name": "warfarin sodium 2.5 MG Oral Tablet", "tty": "SCD"}]},
            {"tty": "SBD", "conceptProperties": [
                {"rxcui": "855290", "name": "warfarin sodium 1 MG Oral Tablet [Coumadin]", "tty": "SBD"},
                {"rxcui": "855292", "name": "warfarin sodium 1 MG Oral Tablet [Jantoven]", "tty": "SBD"},
                {"rxcui": "855298", "name": "warfarin sodium 10 MG Oral Tablet [Coumadin]", "tty": "SBD"},
                {"rxcui": "855300", "name": "warfarin sodium 10 MG Oral Tablet [Jantoven]", "tty": "SBD"}]},
        ]}},
}


def rxnav(url):
    """GET a JSON document from RxNav, falling back to the bundled copy."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json(), "live API"
    except Exception as problem:
        print(f"RxNav unreachable ({type(problem).__name__}: {problem}).")
        if url not in BUNDLED_RESPONSES:
            raise RuntimeError(
                "No network, and no bundled copy of this URL -- only warfarin is bundled. "
                "Set drug_name back to 'warfarin' and re-run from this cell.") from problem
        print("Falling back to the bundled response. Everything below still works.")
        return BUNDLED_RESPONSES[url], "bundled copy"


# TODO (optional, and only if you have a network connection): change this to another
#       ingredient -- metformin, lisinopril, atorvastatin -- and re-run. The bundled
#       fallback only covers warfarin, so leave it as-is if you are offline.
drug_name = "warfarin"

lookup, source = rxnav(f"{RXNAV_BASE}/rxcui.json?name={drug_name}")
rxcui = lookup["idGroup"]["rxnormId"][0]
print(f"{drug_name} -> RxCUI {rxcui}   (source: {source})")

Now ask RxNav what that concept is related to. The response comes back grouped by term type,
which is the terminology telling you which layer of the hierarchy each concept lives on.

In [ ]:
related, source = rxnav(f"{RXNAV_BASE}/rxcui/{rxcui}/related.json?tty=IN+PIN+SCD+SBD")

LAYER_MEANING = {
    "IN": "ingredient -- the concept a drug-allergy rule should fire on",
    "PIN": "precise ingredient -- the salt form",
    "SCD": "clinical drug -- ingredient + strength + form, what gets prescribed",
    "SBD": "branded drug -- the same thing with a trade name on it",
}

rows = []
for group in related["relatedGroup"]["conceptGroup"]:
    concepts = group.get("conceptProperties", [])
    if not concepts:
        continue
    rows.append({"tty": group["tty"],
                 "n_concepts": len(concepts),
                 "example": concepts[0]["name"],
                 "means": LAYER_MEANING.get(group["tty"], "")})

print(f"source: {source}\n")
print(pd.DataFrame(rows).sort_values("tty").to_string(index=False))

This is the same move you made in Part 2 with `descendants_of`, running against a terminology
the whole country prescribes from. An interaction alert written against the `IN` concept for
warfarin fires for every branded tablet underneath it; an alert written against one `SBD`
misses Jantoven, or misses the 2.5 mg tablet, or misses whatever the pharmacy substituted
this morning. Choosing the level of the hierarchy to attach a rule to is the entire
engineering problem, and it is a modeling decision, not a coding one.

## Discussion

1. The `staging_system` conflict has no graph-theoretic solution — the ontology simply does
   not say. Who should resolve it: the terminology maintainer by adding a local override, the
   application developer by picking a branch, or the clinician at the point of care? What
   does each choice cost?
2. We got a six-orders-of-magnitude error by following `part_of` as though it were `is_a`.
   Name another pair of relations in clinical data that look interchangeable and are not —
   and say what the analogous error would look like in a cohort query.
3. Cohort definitions roll up the is-a tree, so adding a new leaf concept to SNOMED CT
   silently changes the size of every cohort above it. If a study's denominator changes
   because the terminology was updated, whose result was wrong — the old one or the new one?

## Solutions

Completed versions of the three TODOs, as markdown so nothing here runs by accident.

**Part 2 — `descendants_of`:**

```python
def descendants_of(concept, relation="is_a"):
    found, queue = [], deque(linked_down(concept, relation))
    while queue:
        current = queue.popleft()
        if current in found:
            continue
        found.append(current)
        queue.extend(linked_down(current, relation))
    return found
```

**Part 2 — `inherit`, made recursive:**

```python
def inherit(concept, slot, relations=("is_a",)):
    if slot in ontology.nodes[concept]:
        return ontology.nodes[concept][slot], concept
    for relation in relations:
        for parent in linked_up(concept, relation):
            value, source = inherit(parent, slot, relations)
            if value is not None:
                return value, source
    return None, None
```

This returns the first answer found in a depth-first walk, which is fine for a tree and is
*not* fine once a concept has two parents — the answer depends on which parent happens to be
listed first in `IS_A`. That is precisely the bug Part 3 exists to expose.

**Part 3 — `resolve`, with the tie detected:**

```python
def resolve(concept, slot):
    found = candidate_values(concept, slot)
    if not found:
        return {"value": None, "source": None, "distance": None, "ambiguous": False}

    nearest = found[0][0]
    tied = [(distance, source, value) for distance, source, value in found
            if distance == nearest]
    distinct_values = {value for _, _, value in tied}

    distance, source, value = tied[0]
    return {"value": value, "source": source, "distance": distance,
            "ambiguous": len(distinct_values) > 1}
```

Two candidates at the same distance carrying the *same* value are not a conflict, which is
why the check is on the set of distinct values rather than on the number of candidates.